# Seismic Ground Motion Prediction: BiLSTM + EBM + ReAct Agent

## Features
| Case | Inputs | Outputs |
|------|--------|---------|
| **Case A** (no depth) | magnitude, log_vs30, rjb, log_rjb, ft_strike_slip, ft_reverse, ft_normal | log(PGA), log(PSA @ all periods) |
| **Case B** (with depth) | + depth | same |

## Key Design Decisions
- **All R² and RMSE computed on log scale** (log of acceleration in g)
- Fault type → one-hot encoded (strike-slip / reverse / normal)
- log_rjb and log_vs30 always kept consistent when rjb or vs30 is varied in parametric studies
- BiLSTM: multi-output (predicts PGA + all SA periods in one forward pass)
- EBM: one per output (more interpretable)
- **NaN-aware loss masking** for SA periods with missing values
- Includes plots required by SDA Report format (data map, hyperparameter tuning,
  std-deviation, attenuation overlaid with observations, SHAP, etc.)

In [ ]:
# ================================================================
#  SEISMIC: BiLSTM + EBM + ReAct Agentic AI
# ================================================================

import re, json, math, random, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap, torch, torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from interpret.glassbox import ExplainableBoostingRegressor
from scipy.stats import skew, kurtosis

# LangChain / Ollama is optional — the agent block guards against it being missing
try:
    from langchain_ollama import ChatOllama
    _HAS_OLLAMA = True
except Exception:
    _HAS_OLLAMA = False
    print("Note: langchain_ollama not available — agent block will be skipped.")

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────
CSV_PATH = r"D:\sd\ESM_filtered.csv"   # ← change to your path
OUT_DIR  = Path("agent_outputs"); OUT_DIR.mkdir(exist_ok=True)

FEATS_A = ["magnitude", "mag2", "log_vs30", "rjb", "log_rjb", "rjb2",
           "mag_log_rjb", "ft_strike_slip", "ft_reverse", "ft_normal"]
FEATS_B = FEATS_A + ["depth"]

SEED, BATCH, EPOCHS, LR, PATIENCE = 42, 32, 20, 1e-3, 5
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def save_fig(fig, name):
    p = OUT_DIR / f"{name}.png"
    fig.savefig(p, dpi=300, bbox_inches="tight")   # 300 dpi per report spec
    plt.show()
    plt.close(fig)
    return str(p)

In [ ]:
# ── METRICS — all on LOG scale ───────────────────────────────────
def metrics_log(y_log_true, y_log_pred):
    """All inputs must already be in log scale."""
    r2   = r2_score(y_log_true, y_log_pred)
    rmse = math.sqrt(mean_squared_error(y_log_true, y_log_pred))
    mae  = mean_absolute_error(y_log_true, y_log_pred)
    bias = float(np.mean(y_log_pred - y_log_true))
    std  = float(np.std(y_log_pred - y_log_true))
    return {"r2": r2, "rmse": rmse, "mae": mae, "bias": bias, "std": std}

In [ ]:
# ── DATA LOADER ──────────────────────────────────────────────────
def load_data(path):
    path = os.path.normpath(path)
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found at: {path}")
    print(f"Loading file from: {path}")
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    print("Columns found:", df.columns.tolist()[:20], "...")

    df = df.rename(columns={
        "Mag": "magnitude",
        "R(km)": "rjb",
        "Vs30(m/s)": "vs30",
        "Depth(km)": "depth",
    })
    for col in ["magnitude", "rjb", "vs30", "depth"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["magnitude", "rjb", "vs30"]).reset_index(drop=True)

    if "U_Hor_0.01" in df.columns:
        df["pga"] = pd.to_numeric(df["U_Hor_0.01"], errors="coerce")
    else:
        raise ValueError("PGA column not found (U_Hor_0.01 missing)")

    df["log_vs30"]    = np.log(df["vs30"].clip(1e-6))
    df["log_rjb"]     = np.log(df["rjb"].clip(1e-6))
    df["rjb2"]        = df["rjb"] ** 2
    df["mag2"]        = df["magnitude"] ** 2
    df["mag_log_rjb"] = df["magnitude"] * df["log_rjb"]

    if "depth" in df.columns:
        df["depth"] = df["depth"].fillna(df["depth"].median())
    else:
        df["depth"] = 10.0  # fallback default

    if "Fault" in df.columns:
        df["fault_type"] = df["Fault"]
    else:
        df["fault_type"] = "unknown"

    df["ft_strike_slip"] = (df["fault_type"] == "strike_slip").astype(float)
    df["ft_reverse"]     = (df["fault_type"] == "reverse").astype(float)
    df["ft_normal"]      = (df["fault_type"] == "normal").astype(float)

    sa_cols, periods = [], []
    for c in df.columns:
        if c.startswith("U_Hor_"):
            try:
                t = float(c.replace("U_Hor_", ""))
                if 0.01 <= t <= 10.0:
                    sa_cols.append(c); periods.append(t)
            except: pass

    sorted_pairs  = sorted(zip(periods, sa_cols))
    periods_clean = [p for p, _ in sorted_pairs]
    sa_cols_clean = [c for _, c in sorted_pairs]

    for c in sa_cols_clean:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    print(f"Loaded {len(df)} rows | {len(sa_cols_clean)} SA periods")
    return df, sa_cols_clean, periods_clean

In [ ]:
# ── DATASET & MODEL ──────────────────────────────────────────────
class SeismicDS(Dataset):
    """Returns (X, Y_filled, mask)  — mask is 1.0 where target was valid, 0.0 where NaN."""
    def __init__(self, X, Y):
        self.X     = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        Y_arr      = np.asarray(Y, dtype=np.float32)
        self.mask  = torch.tensor((~np.isnan(Y_arr)).astype(np.float32))
        Y_filled   = np.where(np.isnan(Y_arr), 0.0, Y_arr)
        self.Y     = torch.tensor(Y_filled, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Y[i], self.mask[i]


class Attention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Sequential(
            nn.Linear(d, d // 2),
            nn.Tanh(),
            nn.Linear(d // 2, 1),
        )
    def forward(self, H):
        w = torch.softmax(self.w(H), dim=1)
        return (w * H).sum(1)


class BiLSTM(nn.Module):
    def __init__(self, n_feats, n_outputs, hidden=64, lstm_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_feats, hidden, lstm_layers,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.attn = Attention(hidden * 2)
        self.fc   = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, n_outputs),
        )
    def forward(self, x):
        H, _ = self.lstm(x)
        return self.fc(self.attn(H))


def masked_mse(pred, target, mask):
    """Mean squared error computed only over valid (non-NaN) target entries."""
    diff2 = (pred - target) ** 2 * mask
    denom = mask.sum().clamp(min=1.0)
    return diff2.sum() / denom

In [ ]:
# ── TRAIN ────────────────────────────────────────────────────────
def train_bilstm(X_tr, Y_tr, X_val, Y_val, n_feats, n_outputs,
                 tag="", hidden=64, lstm_layers=2, dropout=0.2, lr=LR,
                 epochs=EPOCHS, batch=BATCH, patience=PATIENCE, verbose=True):
    sc = StandardScaler().fit(X_tr)
    tr  = DataLoader(SeismicDS(sc.transform(X_tr), Y_tr),  batch, shuffle=True)
    val = DataLoader(SeismicDS(sc.transform(X_val), Y_val), batch)
    model   = BiLSTM(n_feats, n_outputs, hidden=hidden,
                     lstm_layers=lstm_layers, dropout=dropout)
    opt     = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched   = torch.optim.lr_scheduler.ReduceLROnPlateau(
              opt, mode="min", patience=5, factor=0.5)

    best_val, best_state, no_imp = float("inf"), None, 0
    train_hist, val_hist = [], []

    for epoch in range(1, epochs + 1):
        model.train(); ep_loss, ep_n = 0.0, 0
        for X, Y, M in tr:
            l = masked_mse(model(X), Y, M)
            opt.zero_grad(); l.backward(); opt.step()
            ep_loss += l.item() * len(X); ep_n += len(X)
        train_hist.append(ep_loss / max(ep_n, 1))

        model.eval(); v_loss, v_n = 0.0, 0
        with torch.no_grad():
            for X, Y, M in val:
                l = masked_mse(model(X), Y, M)
                v_loss += l.item() * len(X); v_n += len(X)
        vl = v_loss / max(v_n, 1)
        val_hist.append(vl)
        sched.step(vl)
        if verbose:
            print(f"[{tag}] Epoch {epoch:3d}/{epochs}  train={train_hist[-1]:.5f}  val={vl:.5f}")

        if vl < best_val - 1e-6:
            best_val = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
        if no_imp >= patience:
            if verbose: print(f"  Early stop at epoch {epoch}")
            break
    model.load_state_dict(best_state)
    return model, sc, {"train": train_hist, "val": val_hist, "best_val": best_val}


def predict_bilstm_log(model, sc, X):
    model.eval()
    with torch.no_grad():
        Xt = torch.tensor(sc.transform(X), dtype=torch.float32).unsqueeze(1)
        return model(Xt).numpy()

In [ ]:
# ── EBM TRAIN & PREDICT (defined before run_case to avoid NameError) ──
def train_ebms(X_tr, Y_tr_log, output_names):
    """Train one EBM per output column, all on log scale."""
    ebms = {}
    for i, name in enumerate(output_names):
        y_col = Y_tr_log[:, i]
        valid = ~np.isnan(y_col)
        ebm = ExplainableBoostingRegressor(
            random_state=SEED,
            max_rounds=200,
            interactions=5,
            learning_rate=0.02,
            min_samples_leaf=10,
            max_bins=256,
            inner_bags=0,
            outer_bags=4,
        )
        ebm.fit(X_tr[valid], y_col[valid])
        ebms[name] = ebm
        print(f"  EBM trained: {name}")
    return ebms


def predict_ebms_log(ebms, X, output_names):
    preds = []
    for i, name in enumerate(output_names):
        if isinstance(ebms, dict):
            preds.append(ebms[name].predict(X))
        else:
            preds.append(ebms[i].predict(X))
    return np.column_stack(preds)

In [ ]:
# ── BUILD TARGETS (LOG SCALE) ────────────────────────────────────
def build_targets(df, sa_cols_clean):
    pga_log = np.log(df["pga"].clip(1e-9).values)
    sa_log  = np.log(df[sa_cols_clean].clip(1e-9).values)
    return np.column_stack([pga_log, sa_log]).astype(np.float32)

In [ ]:
# ── PLOT HELPERS ─────────────────────────────────────────────────
def _plot_predicted_vs_observed(Y_te, bilstm_preds, ebm_preds, output_names, tag):
    show_idx = [0] + [int(i) for i in np.linspace(1, len(output_names) - 1, 4).round()]
    show_idx = sorted(set(show_idx))[:5]
    fig, axes = plt.subplots(2, len(show_idx), figsize=(5 * len(show_idx), 9))
    fig.suptitle(f"Predicted vs Observed — LOG SCALE [{tag}]",
                 fontsize=13, fontweight="bold")
    for col, i in enumerate(show_idx):
        name   = output_names[i]
        y_true = Y_te[:, i]
        valid  = ~np.isnan(y_true)
        for row, (preds, label, color) in enumerate([
                (bilstm_preds, "BiLSTM", "steelblue"),
                (ebm_preds,    "EBM",    "darkorange")]):
            ax = axes[row][col]
            yt = y_true[valid]; yp = preds[valid, i]
            m  = metrics_log(yt, yp)
            ax.scatter(yt, yp, alpha=0.35, s=8, color=color)
            lo, hi = yt.min(), yt.max()
            ax.plot([lo, hi], [lo, hi], "r--", linewidth=1)
            ax.set_title(f"{label}\n{name}\nR²={m['r2']:.4f} RMSE={m['rmse']:.4f}",
                         fontsize=9)
            ax.set_xlabel("Observed log(g)"); ax.set_ylabel("Predicted log(g)")
            ax.grid(True, alpha=0.3)
    plt.tight_layout()
    save_fig(fig, f"pred_vs_obs_{tag}")


def _plot_residuals(Y_te, bilstm_preds, ebm_preds, df_te, tag):
    i = 0
    for model_name, preds, color in [("BiLSTM", bilstm_preds, "steelblue"),
                                      ("EBM",    ebm_preds,    "darkorange")]:
        y_true = Y_te[:, i]; valid = ~np.isnan(y_true)
        res    = y_true[valid] - preds[valid, i]
        df_v   = df_te.iloc[np.where(valid)[0]]
        cols   = [c for c in ["magnitude", "rjb", "log_vs30", "depth"] if c in df_v.columns]
        n_cols = len(cols) + 1
        fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))
        fig.suptitle(f"Residuals (log scale) — PGA — {model_name} [{tag}]",
                     fontsize=12, fontweight="bold")
        for ax, col in zip(axes[:-1], cols):
            ax.scatter(df_v[col].values, res, alpha=0.35, s=8, color=color)
            ax.axhline(0, color="red", linestyle="--", linewidth=1)
            ax.set_xlabel(col); ax.set_ylabel("Residual (log scale)")
            ax.grid(True, alpha=0.3)
        axes[-1].hist(res, bins=40, color=color, edgecolor="white")
        axes[-1].set_title("Residual Distribution"); axes[-1].grid(True, alpha=0.3)
        plt.tight_layout()
        save_fig(fig, f"residuals_PGA_{model_name}_{tag}")


def _plot_r2_spectrum(results, periods_clean, tag):
    psa_keys = [k for k in results if k.startswith("psa_")]
    periods  = [float(k.split("_")[1]) for k in psa_keys]
    r2_lstm  = [results[k]["bilstm"]["r2"] for k in psa_keys]
    r2_ebm   = [results[k]["ebm"]["r2"]    for k in psa_keys]
    fig, ax  = plt.subplots(figsize=(10, 5))
    ax.semilogx(periods, r2_lstm, "-o", color="steelblue",  label="BiLSTM", markersize=4)
    ax.semilogx(periods, r2_ebm,  "-s", color="darkorange", label="EBM",    markersize=4)
    pga_lstm = results["pga"]["bilstm"]["r2"]
    pga_ebm  = results["pga"]["ebm"]["r2"]
    ax.axhline(pga_lstm, color="steelblue",  linestyle=":", alpha=0.6,
               label=f"BiLSTM PGA R²={pga_lstm:.3f}")
    ax.axhline(pga_ebm,  color="darkorange", linestyle=":", alpha=0.6,
               label=f"EBM PGA R²={pga_ebm:.3f}")
    ax.set_xlabel("Period (s)"); ax.set_ylabel("R² (log scale)")
    ax.set_title(f"R² vs Period — LOG SCALE [{tag}]", fontweight="bold")
    ax.legend(); ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout(); save_fig(fig, f"r2_spectrum_{tag}")


def _plot_spectrum_parametric(bilstm, sc, ebms, feats, output_names,
                               periods_clean, df_tr, tag):
    feat_idx = {f: i for i, f in enumerate(feats)}
    med = np.median(df_tr[feats].values, axis=0)

    def build_X(vary_feat, values):
        X_p = np.tile(med, (len(values), 1)).astype(np.float32)
        X_p[:, feat_idx[vary_feat]] = values
        if vary_feat == "rjb" and "log_rjb" in feat_idx:
            X_p[:, feat_idx["log_rjb"]] = np.log(np.clip(values, 1e-6, None))
        if vary_feat == "vs30" and "log_vs30" in feat_idx:
            X_p[:, feat_idx["log_vs30"]] = np.log(np.clip(values, 1e-6, None))
        return X_p

    scenarios = [
        ("magnitude", np.linspace(4.0, 8.0, 100), "Mw",       [4.0, 5.0, 6.0, 7.0, 7.5]),
        ("rjb",       np.linspace(1.0, 300., 100), "Rjb (km)", [5, 25, 50, 100, 200]),
        ("log_vs30",  np.log(np.linspace(200, 1500, 100)), "ln(Vs30)", None),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Parametric Spectra — LOG SCALE [{tag}]",
                 fontsize=14, fontweight="bold")
    period_arr = np.array(periods_clean)
    psa_names  = [n for n in output_names if n.startswith("psa_")]
    psa_idx    = [output_names.index(n) for n in psa_names]
    colors = plt.cm.plasma(np.linspace(0.1, 0.9, 5))

    for col, (feat, rng, xlabel, legend_vals) in enumerate(scenarios):
        if feat not in feat_idx: continue
        X_p = build_X(feat, rng)
        lstm_log = predict_bilstm_log(bilstm, sc, X_p)[:, psa_idx]
        ebm_log  = predict_ebms_log(ebms, X_p, psa_names)
        rows = np.linspace(0, 99, 5, dtype=int)
        row_labels = legend_vals if legend_vals else [f"{rng[r]:.2f}" for r in rows]
        for ax, (log_preds, model_label) in zip(
                [axes[0][col], axes[1][col]],
                [(lstm_log, "BiLSTM"), (ebm_log, "EBM")]):
            for r, (row, lbl, c) in enumerate(zip(rows, row_labels, colors)):
                sa_vals = np.exp(log_preds[row])
                ax.loglog(period_arr, sa_vals, linewidth=2, color=c, label=str(lbl))
            ax.set_xlabel("Period (s)"); ax.set_ylabel("PSA (g)")
            ax.set_title(f"{model_label} — vs {xlabel}", fontweight="bold")
            ax.legend(fontsize=8, title=feat); ax.grid(True, which="both", alpha=0.3)
            ax.set_xlim([period_arr.min(), period_arr.max()])
    plt.tight_layout(); save_fig(fig, f"parametric_spectra_{tag}")

In [ ]:
# ── CASE RUNNER ──────────────────────────────────────────────────
def run_case(df, feats, sa_cols_clean, periods_clean, case_name):
    print(f"\n{'='*60}\n  CASE: {case_name}\n  Features ({len(feats)}): {feats}\n{'='*60}")
    output_names = ["pga"] + [f"psa_{p:.3f}" for p in periods_clean]
    n_feats   = len(feats)
    n_outputs = len(output_names)

    X = df[feats].values.astype(np.float32)
    Y_log = build_targets(df, sa_cols_clean)
    valid_rows = ~np.isnan(Y_log[:, 0])
    X = X[valid_rows]; Y_log = Y_log[valid_rows]
    df_valid = df[valid_rows].reset_index(drop=True)

    X_tr, X_te, Y_tr, Y_te, df_tr, df_te = train_test_split(
        X, Y_log, df_valid, test_size=0.2, random_state=SEED)

    print("\nTraining EBMs (one per output)...")
    ebms = train_ebms(X_tr, Y_tr, output_names)
    ebm_log_preds = predict_ebms_log(ebms, X_te, output_names)

    print("\nTraining BiLSTM (multi-output, masked loss for NaN SA)...")
    bilstm, sc, hist = train_bilstm(X_tr, Y_tr, X_te, Y_te,
                                     n_feats, n_outputs, tag=case_name)
    bilstm_log_preds = predict_bilstm_log(bilstm, sc, X_te)

    print(f"\n{'─'*55}\n  PERFORMANCE REPORT — {case_name} — LOG SCALE\n{'─'*55}")
    results = {}
    for i, name in enumerate(output_names):
        y_true_log = Y_te[:, i]; valid = ~np.isnan(y_true_log)
        if valid.sum() < 10: continue
        m_ebm  = metrics_log(y_true_log[valid], ebm_log_preds[valid, i])
        m_lstm = metrics_log(y_true_log[valid], bilstm_log_preds[valid, i])
        results[name] = {"ebm": m_ebm, "bilstm": m_lstm}
        print(f"\n  {name.upper()}")
        print(f"  EBM    R²={m_ebm['r2']:.4f}  RMSE={m_ebm['rmse']:.4f}  "
              f"MAE={m_ebm['mae']:.4f}  Bias={m_ebm['bias']:+.4f}")
        print(f"  BiLSTM R²={m_lstm['r2']:.4f}  RMSE={m_lstm['rmse']:.4f}  "
              f"MAE={m_lstm['mae']:.4f}  Bias={m_lstm['bias']:+.4f}")

    best_model = ("BiLSTM" if results["pga"]["bilstm"]["r2"] >
                              results["pga"]["ebm"]["r2"] else "EBM")
    print(f"\n  Best model (PGA R², log scale): {best_model}")

    _plot_predicted_vs_observed(Y_te, bilstm_log_preds, ebm_log_preds,
                                 output_names, case_name)
    _plot_residuals(Y_te, bilstm_log_preds, ebm_log_preds, df_te, case_name)
    _plot_r2_spectrum(results, periods_clean, case_name)
    _plot_spectrum_parametric(bilstm, sc, ebms, feats, output_names,
                               periods_clean, df_tr, case_name)

    return (results, bilstm, sc, ebms, X_te, Y_te, df_te,
            output_names, periods_clean, hist, df_tr)

In [ ]:
# ── SHAP / IMPORTANCE ────────────────────────────────────────────
def run_shap(bilstm, sc, ebms, X_tr, X_te, feats, tag):
    print(f"\nSHAP analysis [{tag}]...")
    ebm_pga = ebms["pga"]
    imps    = ebm_pga.term_importances()[:len(feats)]
    fig, ax = plt.subplots(figsize=(7, 4))
    order = np.argsort(imps)
    ax.barh(np.array(feats)[order], np.array(imps)[order],
            color="darkorange", edgecolor="white")
    ax.set_title(f"EBM Feature Importance — PGA [{tag}]", fontweight="bold")
    ax.grid(True, alpha=0.3); plt.tight_layout()
    save_fig(fig, f"ebm_importance_{tag}")

    def lstm_pga(X):
        return predict_bilstm_log(bilstm, sc, X)[:, 0]
    n_bg = min(50, len(X_tr))
    n_te = min(120, len(X_te))
    exp = shap.KernelExplainer(lstm_pga, X_tr[:n_bg])
    sv  = exp.shap_values(X_te[:n_te], nsamples=80)    # kept at 80 for runtime
    fig = plt.figure(figsize=(8, 5))
    shap.summary_plot(sv, X_te[:n_te], feature_names=feats, show=False)
    plt.title(f"SHAP — BiLSTM PGA (log scale) [{tag}]", fontweight="bold")
    plt.tight_layout(); save_fig(fig, f"shap_bilstm_{tag}")

In [ ]:
# ================================================================
#  MAIN EXECUTION
# ================================================================
print("Loading data...")
df, sa_cols_clean, periods_clean = load_data(CSV_PATH)
df = df.reset_index(drop=True)

print("\n--- Running Case A: No Depth ---")
(results_A, bilstm_A, sc_A, ebms_A,
 X_te_A, Y_te_A, df_te_A,
 out_names_A, periods_A, hist_A, df_tr_A) = run_case(
     df, FEATS_A, sa_cols_clean, periods_clean, case_name="CaseA_no_depth")

run_shap(bilstm_A, sc_A, ebms_A,
         df[FEATS_A].values.astype(np.float32),
         X_te_A, FEATS_A, tag="CaseA_no_depth")

print("\n--- Running Case B: With Depth ---")
(results_B, bilstm_B, sc_B, ebms_B,
 X_te_B, Y_te_B, df_te_B,
 out_names_B, periods_B, hist_B, df_tr_B) = run_case(
     df, FEATS_B, sa_cols_clean, periods_clean, case_name="CaseB_with_depth")

run_shap(bilstm_B, sc_B, ebms_B,
         df[FEATS_B].values.astype(np.float32),
         X_te_B, FEATS_B, tag="CaseB_with_depth")

print("\n" + "="*60)
print("  FINAL COMPARISON — PGA R² on LOG SCALE")
print("="*60)
for case_name, res in [("Case A (no depth)", results_A),
                       ("Case B (with depth)", results_B)]:
    m = res["pga"]
    print(f"\n  {case_name}")
    print(f"    BiLSTM: R²={m['bilstm']['r2']:.4f}  RMSE={m['bilstm']['rmse']:.4f}  "
          f"Bias={m['bilstm']['bias']:+.4f}")
    print(f"    EBM   : R²={m['ebm']['r2']:.4f}  RMSE={m['ebm']['rmse']:.4f}  "
          f"Bias={m['ebm']['bias']:+.4f}")

## Required plot — Distribution of data on map

If `Latitude/Longitude` (station or epicenter) columns exist in the CSV,
this draws a station map. Otherwise it falls back to a Mw–Rjb scatter
labelled as "data distribution".

In [ ]:
# ================================================================
# REQUIRED PLOT 1 — Data distribution map
# ================================================================
def plot_data_map(df):
    # Auto-detect lat/lon columns
    lat_candidates = ["lat", "latitude", "ev_lat", "st_lat",
                      "station_lat", "epi_lat", "EQ_lat"]
    lon_candidates = ["lon", "long", "longitude", "ev_lon", "st_lon",
                      "station_lon", "epi_lon", "EQ_lon"]
    lat_col = next((c for c in df.columns if c.lower() in lat_candidates), None)
    lon_col = next((c for c in df.columns if c.lower() in lon_candidates), None)

    fig, ax = plt.subplots(figsize=(11, 7))
    if lat_col and lon_col:
        sc = ax.scatter(df[lon_col], df[lat_col],
                        c=df["magnitude"], cmap="plasma",
                        s=12, alpha=0.7, edgecolor="k", linewidths=0.2)
        cb = plt.colorbar(sc, ax=ax); cb.set_label("Magnitude ($M_w$)")
        ax.set_xlabel("Longitude (°)"); ax.set_ylabel("Latitude (°)")
        ax.set_title("Distribution of recordings on map (color = $M_w$)",
                     fontweight="bold")
        ax.grid(True, alpha=0.3)
    else:
        # Fallback: pseudo-map (Mw vs Rjb), with note
        sc = ax.scatter(df["rjb"], df["magnitude"],
                        c=np.log10(df["pga"].clip(1e-9)),
                        cmap="plasma", s=12, alpha=0.7,
                        edgecolor="k", linewidths=0.2)
        cb = plt.colorbar(sc, ax=ax); cb.set_label("log10(PGA) [g]")
        ax.set_xscale("log")
        ax.set_xlabel("Joyner-Boore distance, $R_{jb}$ (km)")
        ax.set_ylabel("Magnitude ($M_w$)")
        ax.set_title("Data distribution (Mw vs Rjb) — no lat/lon found",
                     fontweight="bold")
        ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout(); save_fig(fig, "fig_data_map")

plot_data_map(df)

In [ ]:
# ================================================================
# FIG 01 & 02 — Data scatter plots
# ================================================================
def plot_data_distribution(df):
    mag_bins = [(3, 4, 'blue'), (4, 5, 'green'), (5, 6, 'red'),
                (6, 7, 'gold'), (7, 8, 'magenta')]
    fault_colors = {'strike_slip': 'red', 'reverse': 'blue',
                    'normal': 'green', 'unknown': 'gray'}

    fig, axes = plt.subplots(2, 1, figsize=(10, 12))
    fig.suptitle('Fig 01: Data Distribution', fontsize=14, fontweight='bold')

    axes[0].scatter(df['rjb'], df['magnitude'], s=15, facecolors='none',
                    edgecolors='gray', linewidths=0.5, alpha=0.6)
    axes[0].set_xscale('log')
    axes[0].set_xlabel('Joyner-Boore distance, $R_{jb}$ (km)')
    axes[0].set_ylabel('Magnitude ($M_w$)')
    axes[0].grid(True, which='both', alpha=0.3)
    axes[0].set_title('Mw vs Rjb')

    pga_vals = df['pga'].clip(lower=1e-9)
    for lo, hi, color in mag_bins:
        mask = (df['magnitude'] >= lo) & (df['magnitude'] < hi)
        if mask.sum() == 0: continue
        axes[1].scatter(df.loc[mask, 'rjb'], pga_vals[mask],
                        s=10, color=color, alpha=0.6, label=f'${lo}<M_w<{hi}$')
    axes[1].set_xscale('log'); axes[1].set_yscale('log')
    axes[1].set_xlabel('Joyner-Boore distance, $R_{jb}$ (km)')
    axes[1].set_ylabel('PGA (g)')
    axes[1].set_title('PGA vs Rjb colored by magnitude bin')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, which='both', alpha=0.3)
    plt.tight_layout(); save_fig(fig, 'fig01_data_scatter')

    fig2, ax2 = plt.subplots(figsize=(10, 6))
    fig2.suptitle('Fig 02: Mw vs Rjb by Fault Type', fontsize=14, fontweight='bold')
    for ft, color in fault_colors.items():
        mask = df['fault_type'] == ft
        if mask.sum() == 0: continue
        ax2.scatter(df.loc[mask, 'rjb'], df.loc[mask, 'magnitude'],
                    s=10, color=color, alpha=0.6,
                    label=ft.replace('_', ' ').title(), marker='*')
    ax2.set_xscale('log')
    ax2.set_xlabel('Joyner-Boore distance ($R_{jb}$), km')
    ax2.set_ylabel('Magnitude ($M_w$)')
    ax2.legend(fontsize=9); ax2.grid(True, which='both', alpha=0.3)
    plt.tight_layout(); save_fig(fig2, 'fig02_data_faulttype')

plot_data_distribution(df)

In [ ]:
# ================================================================
# FIG 03 — Frequency / Histogram plots (Statistics plot)
# ================================================================
def plot_frequency(df):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('Fig 03: Frequency Plots', fontsize=14, fontweight='bold')
    axes[0].hist(df['magnitude'], bins=20, color='salmon', edgecolor='white')
    axes[0].set_xlabel('Magnitude ($M_w$)'); axes[0].set_ylabel('No. of Records')
    axes[0].set_title('Magnitude Distribution'); axes[0].grid(True, alpha=0.3)
    axes[1].hist(df['rjb'], bins=30, color='salmon', edgecolor='white')
    axes[1].set_xlabel('Joyner-Boore distance, $R_{jb}$ (km)')
    axes[1].set_ylabel('No. of Records')
    axes[1].set_title('Distance Distribution'); axes[1].grid(True, alpha=0.3)
    ft_counts = df['fault_type'].value_counts()
    axes[2].barh(ft_counts.index, ft_counts.values, color='salmon', edgecolor='white')
    axes[2].set_xlabel('No. of Records'); axes[2].set_title('Fault Type Distribution')
    axes[2].grid(True, alpha=0.3)
    plt.tight_layout(); save_fig(fig, 'fig03_frequency')

plot_frequency(df)

In [ ]:
# ================================================================
# FIG 04 — BiLSTM Architecture diagram
# ================================================================
def plot_bilstm_architecture(feats, n_outputs=6):
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.axis('off')
    ax.set_title('Fig 04: BiLSTM Architecture', fontsize=13, fontweight='bold', pad=12)
    output_labels = ['ln(PGA)', 'ln(PSA$_{0.1s}$)', 'ln(PSA$_{0.5s}$)',
                     '...', 'ln(PSA$_{1s}$)', 'ln(PSA$_{4s}$)']
    layer_configs = [
        (feats,         0.05, 'mediumpurple', 'Inputs'),
        ([''] * 8,      0.25, 'lightpink',    'BiLSTM\nLayer 1'),
        ([''] * 6,      0.42, 'khaki',        'BiLSTM\nLayer 2'),
        ([''] * 6,      0.58, 'lightpink',    'Hidden'),
        ([''] * 4,      0.75, 'khaki',        'Attention'),
        (output_labels, 0.92, 'lightgreen',   'Outputs'),
    ]
    layer_nodes = []
    for labels, x, color, title in layer_configs:
        n = len(labels); ys = np.linspace(0.1, 0.9, n); nodes = []
        for y, lbl in zip(ys, labels):
            circ = plt.Circle((x, y), 0.022, color=color, zorder=3, ec='gray', lw=0.8)
            ax.add_patch(circ)
            if lbl:
                ha = 'right' if x < 0.5 else 'left'
                offset = -0.035 if x < 0.5 else 0.035
                ax.text(x + offset, y, lbl, ha=ha, va='center', fontsize=8)
            nodes.append((x, y))
        ax.text(x, 0.96, title, ha='center', va='bottom',
                fontsize=8, fontweight='bold', color='dimgray')
        layer_nodes.append(nodes)
    for i in range(len(layer_nodes) - 1):
        src, dst = layer_nodes[i], layer_nodes[i + 1]
        step_s = max(1, len(src) // 5); step_d = max(1, len(dst) // 5)
        for sx, sy in src[::step_s]:
            for dx, dy in dst[::step_d]:
                ax.plot([sx, dx], [sy, dy], color='gray', alpha=0.15, lw=0.6, zorder=1)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    plt.tight_layout(); save_fig(fig, 'fig04_bilstm_architecture')

plot_bilstm_architecture(FEATS_A)

## Required plot — Hyperparameter tuning

Two figures:

1. **Training curves** (loss vs epoch) for both cases — saved automatically from `hist_A`/`hist_B`.
2. **Grid-search heatmap** — small grid over hidden-units and learning-rate
   on Case A using a quick training run (epochs cut for speed).

In [ ]:
# ================================================================
# REQUIRED PLOT — Hyperparameter tuning
# ================================================================
def plot_training_curves(hist_A, hist_B):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Hyperparameter tuning — Training curves",
                 fontsize=13, fontweight="bold")
    for ax, hist, label in zip(axes, [hist_A, hist_B], ["Case A (no depth)", "Case B (depth)"]):
        ep = np.arange(1, len(hist["train"]) + 1)
        ax.plot(ep, hist["train"], "-o", color="steelblue", label="Train loss", markersize=3)
        ax.plot(ep, hist["val"],   "-s", color="crimson",  label="Val loss",   markersize=3)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Masked MSE (log scale)")
        ax.set_title(label); ax.grid(True, alpha=0.3); ax.legend()
    plt.tight_layout(); save_fig(fig, "fig_hp_training_curves")

plot_training_curves(hist_A, hist_B)


def hp_grid_search(df, feats, sa_cols_clean, periods_clean,
                   hidden_grid=(32, 64, 128),
                   lr_grid=(5e-4, 1e-3, 2e-3),
                   sample_rows=4000, epochs=8):
    """Quick grid-search for the report. Subsamples + short training."""
    df_s = df.sample(min(sample_rows, len(df)), random_state=SEED).reset_index(drop=True)
    output_names = ["pga"] + [f"psa_{p:.3f}" for p in periods_clean]
    X = df_s[feats].values.astype(np.float32)
    Y = build_targets(df_s, sa_cols_clean)
    valid = ~np.isnan(Y[:, 0]); X = X[valid]; Y = Y[valid]
    X_tr, X_te, Y_tr, Y_te = train_test_split(X, Y, test_size=0.2, random_state=SEED)

    grid = np.zeros((len(hidden_grid), len(lr_grid)))
    for i, h in enumerate(hidden_grid):
        for j, lr in enumerate(lr_grid):
            print(f"  HP search: hidden={h}, lr={lr}")
            _, _, hist = train_bilstm(X_tr, Y_tr, X_te, Y_te,
                                       len(feats), len(output_names),
                                       tag=f"h{h}_lr{lr}", hidden=h, lr=lr,
                                       epochs=epochs, patience=3, verbose=False)
            grid[i, j] = hist["best_val"]

    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(grid, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(lr_grid)));  ax.set_xticklabels([f"{lr:.0e}" for lr in lr_grid])
    ax.set_yticks(range(len(hidden_grid))); ax.set_yticklabels([str(h) for h in hidden_grid])
    ax.set_xlabel("Learning rate"); ax.set_ylabel("Hidden units")
    ax.set_title("HP grid search — Best Val Loss (lower = better)", fontweight="bold")
    for i in range(len(hidden_grid)):
        for j in range(len(lr_grid)):
            ax.text(j, i, f"{grid[i, j]:.3f}", ha="center", va="center",
                    color="white" if grid[i, j] > grid.mean() else "black", fontsize=9)
    plt.colorbar(im, ax=ax)
    plt.tight_layout(); save_fig(fig, "fig_hp_gridsearch")
    return grid

# HP grid search is slow (9 short BiLSTM trainings).
# Disabled by default to keep total runtime ~10-15 min on small datasets.
# Uncomment to run for the report:
# hp_grid = hp_grid_search(df, FEATS_A, sa_cols_clean, periods_clean)
print("HP grid search skipped (uncomment in cell to enable). Training curves above are sufficient for most reports.")

## Required plot — Standard deviation across periods

Total sigma is the **std of residuals on log scale**. If event IDs are
present we approximate between-event (τ) and within-event (φ) sigma using
a simple group-mean decomposition:
- τ = std of per-event mean residuals
- φ = sqrt(σ_total² − τ²)

If no event ID column is found, only σ_total is plotted.

In [ ]:
# ================================================================
# REQUIRED PLOT — Standard deviation (total / between / within event)
# ================================================================
def _detect_event_col(df):
    for c in ["event_id", "EQ_id", "earthquake_id", "EventID", "EvID", "eqid", "EQID"]:
        if c in df.columns: return c
    return None

def plot_sigma_decomposition(results, df_te, Y_te, bilstm_preds, ebm_preds,
                              output_names, periods_clean, tag):
    eq_col = _detect_event_col(df_te)
    psa_keys = [k for k in output_names if k.startswith("psa_") and k in results]
    periods  = [float(k.split("_")[1]) for k in psa_keys]

    def decompose(y_true, y_pred, df_v):
        res = y_true - y_pred
        total = float(np.std(res))
        if eq_col is None:
            return total, np.nan, np.nan
        tau_grp = df_v.groupby(eq_col).apply(lambda g: res[g.index - df_v.index[0]].mean()
                                              if len(g) > 1 else np.nan)
        tau = float(np.nanstd(tau_grp))
        phi = float(np.sqrt(max(total**2 - tau**2, 0.0)))
        return total, tau, phi

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Standard deviation across periods (log scale) [{tag}]",
                 fontsize=13, fontweight="bold")

    for ax, (model_name, preds, color) in zip(
            axes, [("BiLSTM", bilstm_preds, "steelblue"),
                   ("EBM",    ebm_preds,    "darkorange")]):
        sig_tot, sig_tau, sig_phi = [], [], []
        for k in psa_keys:
            i = output_names.index(k)
            yt = Y_te[:, i]; valid = ~np.isnan(yt)
            df_v = df_te.iloc[np.where(valid)[0]].reset_index(drop=True)
            t, ta, ph = decompose(yt[valid], preds[valid, i], df_v)
            sig_tot.append(t); sig_tau.append(ta); sig_phi.append(ph)
        ax.semilogx(periods, sig_tot, "-o", color=color, label=r"$\sigma_{total}$")
        if eq_col is not None:
            ax.semilogx(periods, sig_tau, "--^", color="black", label=r"$\tau$ (between)")
            ax.semilogx(periods, sig_phi, "--v", color="gray",  label=r"$\phi$ (within)")
        ax.set_xlabel("Period (s)"); ax.set_ylabel("Sigma (log units)")
        ax.set_title(f"{model_name} — sigma vs period", fontweight="bold")
        ax.grid(True, which="both", alpha=0.3); ax.legend()
        if eq_col is None:
            ax.text(0.02, 0.95, "No event-ID column found; only total sigma shown.",
                    transform=ax.transAxes, fontsize=8, color="dimgray", va="top")
    plt.tight_layout(); save_fig(fig, f"fig_sigma_{tag}")

# Recompute predictions for plotting
bilstm_preds_A = predict_bilstm_log(bilstm_A, sc_A, X_te_A)
ebm_preds_A    = predict_ebms_log(ebms_A, X_te_A, out_names_A)
bilstm_preds_B = predict_bilstm_log(bilstm_B, sc_B, X_te_B)
ebm_preds_B    = predict_ebms_log(ebms_B, X_te_B, out_names_B)

plot_sigma_decomposition(results_A, df_te_A, Y_te_A, bilstm_preds_A, ebm_preds_A,
                          out_names_A, periods_A, "CaseA_no_depth")
plot_sigma_decomposition(results_B, df_te_B, Y_te_B, bilstm_preds_B, ebm_preds_B,
                          out_names_B, periods_B, "CaseB_with_depth")

## Required plot — Attenuation curves with observations

PGA vs Rjb at fixed Mw bands. Observed data points are overlaid on the
model's median attenuation curve, exactly as the report format expects.

In [ ]:
# ================================================================
# REQUIRED PLOT — Attenuation curves overlaid with observations
# ================================================================
def plot_attenuation_with_data(bilstm, sc, ebms, feats, output_names,
                                df, mw_bands=((4.5, 5.5), (5.5, 6.5), (6.5, 7.5)),
                                vs30_fix=760, fault="strike_slip", tag=""):
    feat_idx = {f: i for i, f in enumerate(feats)}
    rjb_grid = np.logspace(0, np.log10(300), 100)

    fig, ax = plt.subplots(figsize=(11, 7))
    fig.suptitle(f"Attenuation: PGA vs $R_{{jb}}$ — model vs observations [{tag}]",
                 fontsize=13, fontweight="bold")
    colors = plt.cm.plasma(np.linspace(0.1, 0.85, len(mw_bands)))

    for (m_lo, m_hi), color in zip(mw_bands, colors):
        m_mid = 0.5 * (m_lo + m_hi)
        # Build feature row at Mw_mid, varying Rjb
        med = np.median(df[feats].values, axis=0)
        X_p = np.tile(med, (len(rjb_grid), 1)).astype(np.float32)
        X_p[:, feat_idx["magnitude"]] = m_mid
        if "mag2" in feat_idx: X_p[:, feat_idx["mag2"]] = m_mid ** 2
        X_p[:, feat_idx["rjb"]]       = rjb_grid
        if "log_rjb" in feat_idx: X_p[:, feat_idx["log_rjb"]] = np.log(rjb_grid)
        if "rjb2" in feat_idx:    X_p[:, feat_idx["rjb2"]]    = rjb_grid ** 2
        if "mag_log_rjb" in feat_idx:
            X_p[:, feat_idx["mag_log_rjb"]] = m_mid * np.log(rjb_grid)
        if "log_vs30" in feat_idx: X_p[:, feat_idx["log_vs30"]] = np.log(vs30_fix)
        for ft in ["ft_strike_slip", "ft_reverse", "ft_normal"]:
            if ft in feat_idx:
                X_p[:, feat_idx[ft]] = 1.0 if ft == f"ft_{fault}" else 0.0

        # Model curves (median)
        pga_lstm = np.exp(predict_bilstm_log(bilstm, sc, X_p)[:, 0])
        pga_ebm  = np.exp(predict_ebms_log(ebms, X_p, output_names)[:, 0])

        ax.loglog(rjb_grid, pga_lstm, "-",  color=color, lw=2.2,
                  label=f"BiLSTM Mw {m_lo}–{m_hi}")
        ax.loglog(rjb_grid, pga_ebm,  "--", color=color, lw=1.6,
                  label=f"EBM Mw {m_lo}–{m_hi}")

        # Observed points in this Mw band
        mask = (df["magnitude"] >= m_lo) & (df["magnitude"] < m_hi)
        ax.scatter(df.loc[mask, "rjb"], df.loc[mask, "pga"].clip(1e-9),
                   s=8, color=color, alpha=0.35, edgecolors="none")

    ax.set_xlabel("Joyner-Boore distance, $R_{jb}$ (km)")
    ax.set_ylabel("PGA (g)")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout(); save_fig(fig, f"fig_attenuation_with_data_{tag}")

plot_attenuation_with_data(bilstm_A, sc_A, ebms_A, FEATS_A, out_names_A,
                            df, tag="CaseA_no_depth")
plot_attenuation_with_data(bilstm_B, sc_B, ebms_B, FEATS_B, out_names_B,
                            df, tag="CaseB_with_depth")

In [ ]:
# ================================================================
# Sensitivity / Parametric plots (per-model)
# ================================================================
def sensitivity_plots(bilstm, sc, ebms, feats, output_names, periods_clean,
                      df_tr, model_label="BiLSTM"):
    feat_idx    = {f: i for i, f in enumerate(feats)}
    med         = np.median(df_tr[feats].values, axis=0)
    periods_arr = np.array(periods_clean)
    psa_names = [n for n in output_names if n.startswith("psa_")]
    psa_idx   = [output_names.index(n) for n in psa_names]

    def build_X(updates):
        X = med.copy()
        for f, v in updates.items():
            X[feat_idx[f]] = v
        if "rjb" in updates and "log_rjb" in feat_idx:
            X[feat_idx["log_rjb"]] = np.log(max(updates["rjb"], 1e-6))
        if "magnitude" in updates and "mag2" in feat_idx:
            X[feat_idx["mag2"]] = updates["magnitude"] ** 2
        if ("magnitude" in updates or "rjb" in updates) and "mag_log_rjb" in feat_idx:
            mag = updates.get("magnitude", med[feat_idx["magnitude"]])
            rjb = updates.get("rjb",       np.exp(med[feat_idx["log_rjb"]]) if "log_rjb" in feat_idx else 10)
            X[feat_idx["mag_log_rjb"]] = mag * np.log(max(rjb, 1e-6))
        return X

    use_lstm = model_label.lower().startswith("bilstm")
    def predict_psa(X_rows):
        X_arr = np.array(X_rows, dtype=np.float32)
        if use_lstm:
            return np.exp(predict_bilstm_log(bilstm, sc, X_arr)[:, psa_idx])
        else:
            return np.exp(predict_ebms_log(ebms, X_arr, psa_names))

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    fig.suptitle(f"Sensitivity Analysis — {model_label}", fontsize=14, fontweight="bold")

    # a) Vary Rjb
    ax = axes[0][0]
    base = {"magnitude": 7.5, "log_vs30": np.log(760),
            "ft_strike_slip": 1.0, "ft_reverse": 0.0, "ft_normal": 0.0}
    for rjb, color in zip([5, 25, 50, 75], ["red", "green", "blue", "black"]):
        psa = predict_psa([build_X({**base, "rjb": rjb})])[0]
        ax.loglog(periods_arr, psa, color=color, lw=2, label=f"$R_{{jb}}$={rjb} km")
    ax.set_xlabel("Period (s)"); ax.set_ylabel("PSA (g)")
    ax.set_title("a) $M_w$ 7.5, $V_{s30}$=760, Strike slip", fontsize=10)
    ax.legend(fontsize=9); ax.grid(True, which="both", alpha=0.3)

    # b) Vary Mw
    ax = axes[0][1]
    base = {"rjb": 20, "log_vs30": np.log(760),
            "ft_strike_slip": 1.0, "ft_reverse": 0.0, "ft_normal": 0.0}
    for mw, color in zip([4.5, 5.5, 6.5, 7.5], ["red", "green", "blue", "black"]):
        psa = predict_psa([build_X({**base, "magnitude": mw})])[0]
        ax.loglog(periods_arr, psa, color=color, lw=2, label=f"$M_w$ {mw}")
    ax.set_xlabel("Period (s)"); ax.set_ylabel("PSA (g)")
    ax.set_title("b) $R_{jb}$=20 km, $V_{s30}$=760, Strike slip", fontsize=10)
    ax.legend(fontsize=9); ax.grid(True, which="both", alpha=0.3)

    # c) Vary Vs30
    ax = axes[1][0]
    base = {"magnitude": 7.5, "rjb": 20,
            "ft_strike_slip": 1.0, "ft_reverse": 0.0, "ft_normal": 0.0}
    for vs, color in zip([180, 360, 760, 1500], ["red", "green", "blue", "black"]):
        psa = predict_psa([build_X({**base, "log_vs30": np.log(vs)})])[0]
        ax.loglog(periods_arr, psa, color=color, lw=2, label=f"$V_{{s30}}$={vs} m/s")
    ax.set_xlabel("Period (s)"); ax.set_ylabel("PSA (g)")
    ax.set_title("c) $M_w$ 7.5, $R_{jb}$=20, Strike slip", fontsize=10)
    ax.legend(fontsize=9); ax.grid(True, which="both", alpha=0.3)

    # d) Vary fault
    ax = axes[1][1]
    base = {"magnitude": 7.5, "rjb": 20, "log_vs30": np.log(760)}
    for label, ft, color in [
        ("Strike slip", {"ft_strike_slip": 1.0, "ft_reverse": 0.0, "ft_normal": 0.0}, "red"),
        ("Normal",      {"ft_strike_slip": 0.0, "ft_reverse": 0.0, "ft_normal": 1.0}, "green"),
        ("Reverse",     {"ft_strike_slip": 0.0, "ft_reverse": 1.0, "ft_normal": 0.0}, "blue"),
    ]:
        psa = predict_psa([build_X({**base, **ft})])[0]
        ax.loglog(periods_arr, psa, color=color, lw=2, label=label)
    ax.set_xlabel("Period (s)"); ax.set_ylabel("PSA (g)")
    ax.set_title("d) $M_w$ 7.5, $R_{jb}$=20, $V_{s30}$=760", fontsize=10)
    ax.legend(fontsize=9); ax.grid(True, which="both", alpha=0.3)

    plt.tight_layout(); save_fig(fig, f"sensitivity_{model_label}")

sensitivity_plots(bilstm_A, sc_A, ebms_A, FEATS_A, out_names_A, periods_A,
                  df_tr_A, model_label="BiLSTM_CaseA")
sensitivity_plots(bilstm_A, sc_A, ebms_A, FEATS_A, out_names_A, periods_A,
                  df_tr_A, model_label="EBM_CaseA")
sensitivity_plots(bilstm_B, sc_B, ebms_B, FEATS_B, out_names_B, periods_B,
                  df_tr_B, model_label="BiLSTM_CaseB")
sensitivity_plots(bilstm_B, sc_B, ebms_B, FEATS_B, out_names_B, periods_B,
                  df_tr_B, model_label="EBM_CaseB")

In [ ]:
# ================================================================
# TABLES 01 / 02 / 03
# ================================================================
def table_data_statistics(df, sa_cols_clean, periods_clean):
    base_cols   = ['magnitude', 'rjb', 'log_rjb', 'log_vs30', 'depth', 'pga']
    base_labels = ['Mw', 'Rjb', 'log(Rjb)', 'log(Vs30)', 'Depth', 'PGA']
    sa_labels   = [f'PSA_{p:.3f}s' for p in periods_clean]
    all_cols    = base_cols + sa_cols_clean
    all_labels  = base_labels + sa_labels
    rows = []
    for col, lbl in zip(all_cols, all_labels):
        if col not in df.columns: continue
        s = df[col].dropna()
        rows.append({'Parameter': lbl,
                     'Min': round(s.min(), 2), 'Max': round(s.max(), 2),
                     'Mean': round(s.mean(), 2), 'Median': round(s.median(), 2),
                     'STD': round(s.std(), 2),
                     'Skewness': round(skew(s), 2),
                     'Kurtosis': round(kurtosis(s), 2)})
    t1 = pd.DataFrame(rows)
    print('\nTable 01: Statistics of the data'); print(t1.to_string(index=False))
    t1.to_csv(OUT_DIR / 'table01_statistics.csv', index=False)
    return t1


def table_residuals(results, df_te, Y_te, preds, output_names,
                     model_key="bilstm", label="CaseA"):
    """Per-period sigma from actual residuals.
       Between/within decomposition uses event-ID grouping if available;
       otherwise those columns are NaN (instead of made-up multipliers)."""
    eq_col = _detect_event_col(df_te)
    rows = []
    for name, m in results.items():
        i = output_names.index(name)
        yt = Y_te[:, i]; valid = ~np.isnan(yt)
        if valid.sum() < 10: continue
        res = yt[valid] - preds[valid, i]
        sig_total = float(np.std(res))
        if eq_col is not None:
            df_v = df_te.iloc[np.where(valid)[0]].reset_index(drop=True)
            df_v["_res"] = res
            tau_grp = df_v.groupby(eq_col)["_res"].mean()
            tau = float(np.std(tau_grp))
            phi = float(np.sqrt(max(sig_total**2 - tau**2, 0.0)))
        else:
            tau = phi = float("nan")
        rows.append({'Parameter': name,
                     'Bias': round(m[model_key]['bias'], 4),
                     'Total sigma':       round(sig_total, 4),
                     'Between-event tau': round(tau, 4) if not np.isnan(tau) else 'n/a',
                     'Within-event phi':  round(phi, 4) if not np.isnan(phi) else 'n/a',
                     'RMSE': round(m[model_key]['rmse'], 4)})
    t2 = pd.DataFrame(rows)
    print(f'\nTable 02: Residuals/Standard deviations [{label}, {model_key}]')
    print(t2.to_string(index=False))
    t2.to_csv(OUT_DIR / f'table02_residuals_{label}_{model_key}.csv', index=False)
    return t2


def table_performance(results, label='CaseA', model_key="bilstm"):
    rows = []
    for name, metrics in results.items():
        r2   = metrics[model_key]['r2']
        bias = metrics[model_key]['bias']
        R    = round(float(np.sqrt(max(r2, 0))), 3)
        rows.append({'Parameter': name, 'R': R, 'R2': round(r2, 3),
                     'Bias': round(bias, 3),
                     'RMSE': round(metrics[model_key]['rmse'], 4),
                     'MAE':  round(metrics[model_key]['mae'],  4)})
    t3 = pd.DataFrame(rows)
    print(f'\nTable 03: Performance parameters [{label}, {model_key}]')
    print(t3.to_string(index=False))
    t3.to_csv(OUT_DIR / f'table03_performance_{label}_{model_key}.csv', index=False)
    return t3


t1 = table_data_statistics(df, sa_cols_clean, periods_clean)
t2_A_lstm = table_residuals(results_A, df_te_A, Y_te_A, bilstm_preds_A,
                             out_names_A, "bilstm", "CaseA")
t2_A_ebm  = table_residuals(results_A, df_te_A, Y_te_A, ebm_preds_A,
                             out_names_A, "ebm",    "CaseA")
t2_B_lstm = table_residuals(results_B, df_te_B, Y_te_B, bilstm_preds_B,
                             out_names_B, "bilstm", "CaseB")
t2_B_ebm  = table_residuals(results_B, df_te_B, Y_te_B, ebm_preds_B,
                             out_names_B, "ebm",    "CaseB")
t3_A_lstm = table_performance(results_A, "CaseA", "bilstm")
t3_A_ebm  = table_performance(results_A, "CaseA", "ebm")
t3_B_lstm = table_performance(results_B, "CaseB", "bilstm")
t3_B_ebm  = table_performance(results_B, "CaseB", "ebm")
print("\nAll tables saved to agent_outputs/")

In [ ]:
# ================================================================
#  REACT AGENT (only runs if langchain_ollama is installed)
# ================================================================
session_memory = []

def seismic_agent(user_query, max_steps=6):
    if not _HAS_OLLAMA:
        print("Skipping agent: langchain_ollama is not installed.")
        return "agent_unavailable"
    llm = ChatOllama(model="llama3")
    memory_str = "\n".join(
        [f"Q: {m['query']} → {m['result']}" for m in session_memory[-3:]]
    ) or "None"
    def fmt(res, case):
        if res is None: return f"{case}: not yet run"
        m = res.get("pga", {})
        return (f"{case} — PGA: BiLSTM R²={m['bilstm']['r2']:.4f} | "
                f"EBM R²={m['ebm']['r2']:.4f} (all log scale)")
    metrics_ctx = fmt(results_A, "Case A (no depth)") + "\n" + fmt(results_B, "Case B (depth)")

    TOOL_DEFS = '''
compare_models()              — R² / RMSE table for both cases, both models
residual_analysis(case)       — residual plots
parametric_study(case)        — PSA spectrum vs magnitude/distance/vs30
shap_analysis(case)           — SHAP importance plots
r2_spectrum(case)             — R² across all SA periods
fault_analysis(case)          — PGA breakdown by fault type
general_answer(question)      — engineering interpretation
finish(answer)                — conclude
'''
    def _agent_compare_models():
        lines = ["Model comparison (R², RMSE on LOG SCALE):"]
        for case_name, res in [("Case A (no depth)", results_A),
                               ("Case B (with depth)", results_B)]:
            if res is None: continue
            m = res.get("pga", {})
            lines.append(f"\n  {case_name} — PGA")
            lines.append(f"    BiLSTM: R²={m['bilstm']['r2']:.4f}  RMSE={m['bilstm']['rmse']:.4f}")
            lines.append(f"    EBM   : R²={m['ebm']['r2']:.4f}  RMSE={m['ebm']['rmse']:.4f}")
        return "\n".join(lines)
    tools_dispatch = {
        "compare_models":    lambda **kw: _agent_compare_models(),
        "residual_analysis": lambda case="A", **kw: f"Residual plots saved for Case {case}.",
        "parametric_study":  lambda case="A", **kw: f"Parametric spectra saved for Case {case}.",
        "shap_analysis":     lambda case="A", **kw: f"SHAP plots saved for Case {case}.",
        "r2_spectrum":       lambda case="A", **kw: f"R² spectrum plot saved for Case {case}.",
        "fault_analysis":    lambda case="A", **kw: f"Fault analysis plots already saved for Case {case}.",
        "general_answer":    lambda question="", **kw: llm.invoke(
            f"Seismic engineering expert. Context: {metrics_ctx}\nQuestion: {question}"
        ).content,
    }

    print(f"\n{'='*60}\n USER: {user_query}\n{'='*60}")
    history = []
    for step in range(max_steps):
        prompt = f'''You are a seismic AI assistant. Use tools step by step.
ALL metrics are on LOG SCALE.

Tools available:
{TOOL_DEFS}

{metrics_ctx}
Memory: {memory_str}
History: {json.dumps(history)}
Query: {user_query}

Respond ONLY with JSON: {{"thought":"...","tool":"...","args":{{"key":"value"}}}}'''
        raw = llm.invoke(prompt).content.strip()
        try:
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            dec = json.loads(m.group())
            tool_name = dec["tool"]; args = dec.get("args", {})
            print(f"\n  Step {step+1} | THOUGHT: {dec.get('thought','')}")
            print(f"  TOOL: {tool_name}({args})")
            if tool_name == "finish":
                answer = args.get("answer", "Analysis complete.")
                print(f"\n  FINAL: {answer}")
                session_memory.append({"query": user_query, "result": answer})
                return answer
            result = tools_dispatch.get(tool_name, lambda **kw: f"Unknown tool: {tool_name}")(**args)
            print(f"  RESULT: {result}")
            history.append({"step": step+1, "tool": tool_name, "result": str(result)})
        except Exception as e:
            print(f"  Agent error: {e}\n  Raw: {raw}"); break
    session_memory.append({"query": user_query, "result": "incomplete"})
    return "incomplete"

# Run agent queries (skipped silently if Ollama unavailable)
seismic_agent("Compare both cases and models. Show R² and RMSE on log scale side by side.")
seismic_agent("Which model performs better for PGA prediction and why? Does depth help?")
seismic_agent("Show residual analysis for Case A and explain any patterns with magnitude and distance.")

In [ ]:
# ── List of generated figures ────────────────────────────────────
import os
print("Files saved to agent_outputs/:")
for f in sorted(os.listdir(OUT_DIR)):
    print(" •", f)